In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

dataDir = Path("data")
csvFile = dataDir / "pp-2020.csv"


columns = [
    "transaction_id",
    "price",
    "date_of_transfer",
    "postcode",
    "property_type",
    "new_build",
    "duration",
    "paon",
    "saon",
    "street",
    "locality",
    "town_city",
    "district",
    "county",
    "ppd_category",
    "record_status",
]

allYears = pd.read_csv(csvFile, header=None, names=columns)

# Clean types
allYears["date_of_transfer"] = pd.to_datetime(allYears["date_of_transfer"])
allYears["price"] = pd.to_numeric(allYears["price"])

In [ ]:
allYears.shape

In [ ]:
allYears.isna().sum().sort_values(ascending=False).head()

In [ ]:
allYears.describe()

In [ ]:
cleanData = allYears[allYears["ppd_category"] == "A"].copy()

In [ ]:
columnsToDrop = [
    "transaction_id",
    "paon",
    "saon",
    "street",
    "locality",
    "record_status",
    "county",
    "ppd_category"
]

cleanData.drop(columns=columnsToDrop, inplace=True)

In [ ]:
cleanData.isna().sum().sort_values(ascending=False).head()

In [ ]:
cleanData.describe()

In [ ]:
cleanData = cleanData.dropna(subset=["postcode"])

In [ ]:
cleanData.isna().sum().sort_values(ascending=False).head()

In [ ]:
cleanData = cleanData[(cleanData["price"] >= 90_000) & (cleanData["price"] <= 5_000_000)]

In [ ]:
cleanData.describe()

In [ ]:
cleanData["year"] = cleanData["date_of_transfer"].dt.year
cleanData["month"] = cleanData["date_of_transfer"].dt.month

In [ ]:
cleanData["postcode"] = cleanData["postcode"].str.split().str[0]

In [ ]:
cleanData.head()

In [ ]:
categoricalColumns = [
    "property_type",
    "new_build",
    "duration",
    "postcode",
    "town_city",
    "district",
]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded = encoder.fit_transform(cleanData[categoricalColumns])

encodedData = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(categoricalColumns),
    index=cleanData.index
)

cleanData = pd.concat(
    [cleanData.drop(columns=categoricalColumns), encodedData],
    axis=1
)

In [ ]:
target = "price"

X = cleanData.drop(columns=[target])
y = cleanData[target]

In [ ]:
XTrain, XTest, yTrain, yTest = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

XTrain.shape, XTest.shape

In [ ]:
linReg = LinearRegression()
linReg.fit(XTrain, yTrain)

In [ ]:
yPred = linReg.predict(XTest)
yPred[:10]

In [ ]:
mae = mean_absolute_error(yTest, yPred)
rmse = mean_squared_error(yTest, yPred) ** 0.5
r2 = r2_score(yTest, yPred)

print(f"MAE:  £{mae:,.0f}")
print(f"RMSE: £{rmse:,.0f}")
print(f"R²:   {r2:.3f}")
